# Final evaluation and error analysis — Step 13

**CSE437 Data Science | Group 15 | Owner: Sadat | Next: Step 14**

This notebook inspects the completed official held-out evaluation. The selected-feature Logistic Regression pipeline was frozen in Step 12 (`C=1`, `class_weight=balanced`, threshold 0.5), fitted on all 95,415 development rows, and evaluated once on 23,795 later-arrival test rows. The original dataset, target, problem/questions and split are unchanged.

The official evaluation is implemented in `src/final_evaluation.py`; its protocol was saved before test-label access. The cells below load and verify the resulting artifacts rather than select or change a model. To regenerate the evidence from the frozen pipeline, run `python -m src.final_evaluation` from the repository root. A rerun is reproducibility verification, not a new opportunity to choose settings.

**Execution provenance:** all six code cells below executed sequentially in a fresh Python process with actual stdout captured. A fresh Jupyter-kernel run and canonical nbformat validation remain final submission gates.


In [1]:
from pathlib import Path
import sys, json, hashlib, joblib
import pandas as pd
ROOT=Path.cwd().resolve()
if not (ROOT/"src").is_dir() and (ROOT.parent/"src").is_dir(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
OUT=ROOT/"data/results/step13"
summary=json.loads((OUT/"evaluation_summary.json").read_text())
metrics=pd.read_csv(OUT/"final_metrics.csv").iloc[0]
groups=pd.read_csv(OUT/"subgroup_metrics.csv")
probability=pd.read_csv(OUT/"probability_diagnostics.csv")
errors=pd.read_csv(OUT/"error_examples.csv")
coefficients=pd.read_csv(OUT/"feature_coefficients.csv")
print(json.dumps({"selection":summary["selection"],"official_test_evaluations":summary["official_test_evaluations"],"development_rows_fitted":summary["development_rows_fitted"],"test_rows_evaluated":summary["test_rows_evaluated"]},indent=2))


{
  "selection": {
    "family": "logistic_regression",
    "representation": "selected",
    "search_parameters": {
      "model__C": 1.0,
      "model__class_weight": "balanced"
    },
    "threshold": 0.5,
    "mean_development_f1": 0.7321017246390454
  },
  "official_test_evaluations": 1,
  "development_rows_fitted": 95415,
  "test_rows_evaluated": 23795
}


## Frozen final-test result

The test window is 2017-04-23 through 2017-08-31. It was excluded from feature/model/tuning decisions. Mean development F1 (0.732102) is shown only as context; the single held-out result is not a confidence interval or proof of future performance.

![Final test performance and confusion matrix](../figures/09_final_test_performance.png)


In [2]:
print(metrics[["f1","accuracy","precision","recall","roc_auc","brier_score","test_rows","test_cancellations","test_cancellation_rate","tn","fp","fn","tp"]].to_string())
assert int(metrics.tn+metrics.fp+metrics.fn+metrics.tp)==int(metrics.test_rows)==23795
assert int(metrics.fn+metrics.tp)==int(metrics.test_cancellations)
print("\nDevelopment mean F1:",round(summary["selection"]["mean_development_f1"],6))
print("Held-out F1:",round(metrics.f1,6))
print("Model/threshold reselected after test:",summary["model_reselected_from_test"],summary["threshold_changed_after_test"])


f1                            0.750592
accuracy                      0.760874
precision                     0.654187
recall                        0.880321
roc_auc                       0.875977
brier_score                   0.160007
test_rows                 23795.000000
test_cancellations         9726.000000
test_cancellation_rate        0.408741
tn                         9543.000000
fp                         4526.000000
fn                         1164.000000
tp                         8562.000000

Development mean F1: 0.732102
Held-out F1: 0.750592
Model/threshold reselected after test: False False


## Where the model fails

Balanced weighting favors detecting cancellations: recall is high, but false positives exceed false negatives. Subgroup scores below are descriptive after test access and cannot justify changing the frozen pipeline. Counts depend on group size; small groups must not be overinterpreted.

The fixed-bin probability curve lies below the diagonal: predicted probabilities exceed observed rates in every populated bin. This is consistent with balanced class weighting and means these outputs should not be presented as calibrated probabilities. No post-test calibration or threshold adjustment is performed.

![Probability and hotel error diagnostics](../figures/10_final_error_analysis.png)


In [3]:
for dimension in ["hotel","lead_time_band","deposit_type","market_segment","customer_type"]:
    view=groups.loc[groups.dimension.eq(dimension),["group","rows","cancellations","error_rate","f1","precision","recall","fp","fn"]]
    print("\n"+dimension+":")
    print(view.round(6).to_string(index=False))
assert groups.groupby("dimension").rows.sum().eq(23795).all()
print("\nEvery subgroup dimension reconciles to all 23,795 test bookings.")



hotel:
       group  rows  cancellations  error_rate       f1  precision   recall   fp  fn
Resort Hotel  7459           2666    0.202440 0.741084   0.682565 0.810578 1005 505
  City Hotel 16336           7060    0.255877 0.753857   0.645132 0.906657 3521 659

lead_time_band:
  group  rows  cancellations  error_rate       f1  precision   recall   fp  fn
  31–90  3995           1571    0.275344 0.709456   0.606321 0.854870  872 228
181–365  5769           2897    0.242676 0.792777   0.693962 0.924405 1181 219
 91–180  7564           3617    0.251719 0.781401   0.668172 0.940835 1690 214
    0–7  2388            259    0.175042 0.334395   0.284553 0.405405  264 154
   8–30  3011            933    0.272003 0.588235   0.553977 0.627010  471 348
   366+  1068            449    0.045880 0.948148   0.903226 0.997773   48   1

deposit_type:
     group  rows  cancellations  error_rate       f1  precision   recall   fp   fn
No Deposit 21482           7415    0.264454 0.687840   0.580397 0.844100

## Concrete wrong predictions

The protocol selects, separately for false positives and false negatives, five most-confident errors and five additional errors closest to 0.5. These are deliberately diagnostic slices, not random or representative samples. `error_examples.csv` includes all 20 rows; no reservation-status leakage fields are exported.


In [4]:
print(errors.to_string(index=False))
assert len(errors)==20 and not errors.source_row_id.duplicated().any()
print("\nError-type counts:")
print(errors.groupby(["error_type","example_reason"]).size().to_string())


         example_reason  source_row_id arrival_date        hotel  lead_time lead_time_band deposit_type market_segment   customer_type  previous_cancellations  total_of_special_requests  actual  predicted  cancellation_probability error_type
      most_confident_FP          14182   2017-06-09 Resort Hotel         80          31–90   No Deposit         Direct       Transient                       1                          0       0          1                  0.999603         FP
      most_confident_FP         117788   2017-08-04   City Hotel        211        181–365   No Deposit      Online TA       Transient                       0                          0       0          1                  0.992440         FP
      most_confident_FP         119068   2017-08-28   City Hotel        311        181–365   No Deposit      Online TA       Transient                       0                          0       0          1                  0.990008         FP
      most_confident_FP         

## Model inspection and limits

The strongest positive coefficient is the Non Refund indicator; many large coefficients are particular agent categories. Required parking spaces has a large negative coefficient. These are associations in an encoded linear model—not causal effects or universally comparable feature importance. Sparse categories, source timing, repeated profiles and temporal change can make coefficients unstable.

The full-development fit retains 406 encoded columns. Because the selection rule is refitted on all development rows, that width need not match individual CV-fold widths. `models/final_logistic_regression.joblib` contains the entire fitted preprocessing, selection and classifier pipeline.


In [5]:
model_path=ROOT/"models/final_logistic_regression.joblib"
model=joblib.load(model_path)
assert model.named_steps["model"].get_params(deep=False)["C"]==1.0
assert model.named_steps["model"].get_params(deep=False)["class_weight"]=="balanced"
assert len(coefficients)==summary["encoded_features"]==406
assert hashlib.sha256(model_path.read_bytes()).hexdigest()==summary["output_sha256"]["models/final_logistic_regression.joblib"]
print(coefficients.head(20).to_string(index=False))
print("\nSaved pipeline verified:",model_path.relative_to(ROOT))


                             feature  coefficient  absolute_coefficient                 direction
categorical__deposit_type_Non Refund     2.929658              2.929658 higher cancellation score
               categorical__agent_89    -2.783724              2.783724  lower cancellation score
              categorical__agent_152    -2.744587              2.744587  lower cancellation score
               categorical__agent_17     2.582659              2.582659 higher cancellation score
               categorical__agent_11    -2.566355              2.566355  lower cancellation score
numeric__required_car_parking_spaces    -2.290799              2.290799  lower cancellation score
               categorical__agent_69    -2.138171              2.138171  lower cancellation score
              categorical__agent_201    -2.114686              2.114686  lower cancellation score
              categorical__agent_308    -2.061380              2.061380  lower cancellation score
              catego

## Handoff — Step 14, Sadat

Use the verified EDA, selection, model-comparison, tuning and final-test evidence to answer the three approved research questions. Distinguish descriptive associations from model coefficients, development comparisons from final test performance, and best-evaluated from universally best. Do not retune, recalibrate, change the threshold or switch models after seeing the test result. Step 15 then completes the report, genuine contribution statement, references, raw-data provenance, clean-environment/fresh-kernel checks and final PDF.


In [6]:
for relative,expected in summary["output_sha256"].items():
    actual=hashlib.sha256((ROOT/relative).read_bytes()).hexdigest()
    assert actual==expected,relative
assert summary["official_test_evaluations"]==1
assert summary["representation_unchanged_after_test_prediction"]
assert not summary["model_reselected_from_test"] and not summary["threshold_changed_after_test"]
print("Step 13 evidence hashes verified. Official test result is frozen.")
print("Step 14 — Sadat: answer the three unchanged research questions from measured evidence.")


Step 13 evidence hashes verified. Official test result is frozen.
Step 14 — Sadat: answer the three unchanged research questions from measured evidence.
